### Middleware
Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following:
- Tracking agent behaviour with logging, analytics and debugging.
- Transforming prompts, tools selection and output formatting.
- Adding retries, fallbacks and early termination logic.
- Applying rate limits, guardrails and PII detection.


In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

### Summarization Middleware

In [3]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

### Messagebased summarization
agent = create_agent(
    model="groq:openai/gpt-oss-120b",
    checkpointer=InMemorySaver(),
    middleware=[SummarizationMiddleware(
        max_tokens=100,
        model="groq:openai/gpt-oss-120b",
        trigger= ("messages", 10),
        keep=("messages", 4))
    ],
)

In [4]:
### Run with thread id
config = {"configurable":{"thread_id": "test-1"}}

In [5]:
# Alternative test data
questions = [
    "What is 2+2?",
    "What is 2+3?",
    "What is 2+4?",
    "What is 2+5?",
    "What is 2+6?",
    "What is 2+7?",
    "What is 2+8?",
    "What is 2+9?",
    "What is 2+10?",
]

for q in questions:
    response = agent.invoke({"messages": [HumanMessage(content=q)]}, config)
    print("Messages: ", response["messages"])
    print("Messages length: ", len(response["messages"]))

Messages:  [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='35a2fcd7-d798-4c6e-8a83-cded9fe8120e'), AIMessage(content='2\u202f+\u202f2\u202f=\u202f4.', additional_kwargs={'reasoning_content': 'The user asks a simple math question. The assistant should answer "4". No policy issues.'}, response_metadata={'token_usage': {'completion_tokens': 39, 'prompt_tokens': 78, 'total_tokens': 117, 'completion_time': 0.081111901, 'completion_tokens_details': {'reasoning_tokens': 20}, 'prompt_time': 0.003552439, 'prompt_tokens_details': None, 'queue_time': 0.247910519, 'total_time': 0.08466434}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_9241e9962b', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a21863-1f69-7786-8928-2dd572c736e1-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 78, 'output_tokens': 39, 'total_tokens': 117, 'output_token_details': {'reason

### Token size summarization

In [6]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage
from langchain.tools import tool

@tool
def search_hotels(city:str)->str:
    """Search for hotels in a city"""
    return f"""Hotel in {city}
    1. Grand Hotel - $100
    2. Best Hotel - $200
    3. Cheap Hotel - $300
    """

agent = create_agent(
    model="groq:openai/gpt-oss-120b",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[SummarizationMiddleware(
        max_tokens=100,
        model="groq:openai/gpt-oss-120b",
        trigger= ("tokens", 550),
        keep=("tokens", 200))
    ],
)

config = {"configurable":{"thread_id": "test-2"}}

def count_tokens(messages):
    total_chars = sum(len(m.content) for m in messages)
    return total_chars

In [7]:
# Run test 
cities = ["New York", "Los Angeles", "Chicago", "Houston", "Miami"]
for city in cities:
    response = agent.invoke({"messages": [HumanMessage(content=f"What are the hotels in {city}?")]}, config)
    tokens = count_tokens(response["messages"])
    print(f"City: {city} - Tokens: {tokens}, {len(response["messages"])} messages")
    print(f"{response['messages']}")

City: New York - Tokens: 570, 4 messages
[HumanMessage(content='What are the hotels in New York?', additional_kwargs={}, response_metadata={}, id='8dcbc63e-ef41-4322-b780-7289e91566a9'), AIMessage(content='', additional_kwargs={'reasoning_content': 'The user asks: "What are the hotels in New York?" We need to provide a list of hotels. We can use the provided function search_hotels with city parameter. Let\'s call it.', 'tool_calls': [{'id': 'fc_a4201b18-513b-44db-b894-033613d77df5', 'function': {'arguments': '{"city":"New York"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 69, 'prompt_tokens': 129, 'total_tokens': 198, 'completion_time': 0.143142778, 'completion_tokens_details': {'reasoning_tokens': 40}, 'prompt_time': 0.005138072, 'prompt_tokens_details': None, 'queue_time': 0.37521743, 'total_time': 0.14828085}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_86dc843b12', 'service_tier': 'on_demand', 'finish

### Human In The Loop Middleware
Pause agent execution for Human approval, editing, or rejecting of tool calls before they execute. Human-in-the-loop is useful for the following:
- High stakes operations requiring human approval (e.g. database writes, financial transactions)
- Compliance workflows where human oversight is mandatory 
- Long running conversations where human feedback guides the agent

In [8]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id:str)->str:
    """Mock function to read an email by its id"""
    return f"Email content for ID:{email_id}"

def send_email_tool(recipient:str, subject:str, body:str)->str:
    """Mock function to send an email"""
    return f"Email sent to {recipient} with subject {subject} and body {body}"

In [9]:
agent = create_agent(
    model="groq:openai/gpt-oss-120b",
    tools=[read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {
                    "allowed_decisions":["approve", "edit", "reject"]
                },
                "read_email_tool": False,
            }
        )
    ],
)

In [24]:
config = {"configurable":{"thread_id": "test-approve"}}
# Step 1: Request 
result = agent.invoke(
    {
      "messages": [
            HumanMessage(content="Send an email to john@example.com with the subject 'Meeting reminder' and body 'Don't forget the meeting tomorrow'")
        ],
    },
    config=config,
)

In [25]:
result

{'messages': [HumanMessage(content="Send an email to john@example.com with the subject 'Meeting reminder' and body 'Don't forget the meeting tomorrow'", additional_kwargs={}, response_metadata={}, id='2a76f60b-a82c-4129-9d95-4d7ecfb44300'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'User wants to send an email. Use send_email_tool.', 'tool_calls': [{'id': 'fc_f9c6083d-f96d-4b6d-a5b9-d1235e002f8d', 'function': {'arguments': '{"body":"Don\'t forget the meeting tomorrow","recipient":"john@example.com","subject":"Meeting reminder"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 62, 'prompt_tokens': 179, 'total_tokens': 241, 'completion_time': 0.130366451, 'completion_tokens_details': {'reasoning_tokens': 13}, 'prompt_time': 0.010782772, 'prompt_tokens_details': None, 'queue_time': 0.342522033, 'total_time': 0.141149223}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_49bfac06f1', 'service_ti

In [26]:
# Step 2: Approve
from langgraph.types import Command


if "__interrupt__" in result:
    print("Paused! Human intervention required.")
    result = agent.invoke(
        Command(
            resume= {
                "decisions": [
                    {"type":"approve"},
                ]
            }
        ),
        config=config,
    )

print(f"Result: {result['messages'][-1].content}")

Paused! Human intervention required.
Result: The email has been sent again to **john@example.com** with the subject **“Meeting reminder”** and the body **“Don’t forget the meeting tomorrow.”** Let me know if there’s anything else you’d like to do!
